# 🤖 Phase 2: TF-IDF Classifier (Naive Bayes & Logistic Regression)

## 🏗 Overview: The Count-Based Representation (TF-IDF)
In this phase, we build our initial classifiers using the **TF-IDF (Term Frequency-Inverse Document Frequency)** representation established in Phase 1.

### 🔎 Rationale: Comparing "Chefs" on the Same "Ingredients"
1.  **Naive Bayes**: A probabilistic probabilistic classifier. Excellent for counting and likelihoods.
2.  **Logistic Regression**: A linear classifier that draws a boundary through the 5,000-dimensional TF-IDF space to separate Real from Fake news.

In [ ]:
import os
import pandas as pd
import joblib
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Colab Integration Check
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

## 1. Data Ingestion
Loading the stratified sets and the fitted TF-IDF vectorizer.

In [ ]:
train_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/test.csv'))
tfidf = joblib.load(os.path.join(BASE_PATH, 'models/vectorizer.joblib'))

X_train = tfidf.transform(train_df['cleaned_text'].astype(str))
y_train = train_df['label']

X_test = tfidf.transform(test_df['cleaned_text'].astype(str))
y_test = test_df['label']
print("✅ Data & Vectors Loaded.")

## 2. Model 1 Training: Multinomial Naive Bayes
The probabilistic baseline.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
joblib.dump(nb_model, os.path.join(BASE_PATH, 'models/nb_tfidf_classifier.joblib'))
print("✅ Naive Bayes Trained & Saved.")

## 3. Model 2 Training: Logistic Regression
The linear baseline (drawing a line in the TF-IDF space).

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
joblib.dump(lr_model, os.path.join(BASE_PATH, 'models/lr_tfidf_classifier.joblib'))
print("✅ Logistic Regression (TF-IDF) Trained & Saved.")

## 4. Initial Comparison
How do the two different 'Chefs' compare on the same TF-IDF ingredients?

In [ ]:
nb_pred = nb_model.predict(X_test)
lr_pred = lr_model.predict(X_test)

print("--- Naive Bayes Metrics ---")
print(classification_report(y_test, nb_pred))

print("\n--- Logistic Regression Metrics ---")
print(classification_report(y_test, lr_pred))

## 5. Final Deliverable: Validation Predictions
Predicting for `validation_data.csv` using the best TF-IDF model, ensuring NO extra columns.

In [ ]:
val_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/validation_data.csv'))

def clean_text(text):
    import re, string
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    if pd.isna(text): return ""
    text = text.lower()
    pattern = re.compile('[%s]' % re.escape(string.punctuation))
    text = pattern.sub('', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return " ".join(filtered_tokens)

val_full_text = (val_df['title'] + " " + val_df['text']).apply(clean_text)
X_val = tfidf.transform(val_full_text)

val_df['label'] = lr_model.predict(X_val) # Using LR as it usually outperforms NB in accuracy
final_output_path = os.path.join(BASE_PATH, 'dataset/validation_results.csv')
val_df.to_csv(final_output_path, index=False)
print(f"✅ Validation results saved to: {final_output_path}")